In [ ]:
# exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol.py
# Single-column friendly version for IJCAI:
#   - 3x1 vertical panels (each panel uses full column width)
#   - bigger fonts/lines/markers
#   - shared legend at bottom (larger, multi-column)
#
# Run:
#   python exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol.py

import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")  # helps some WSL setups

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib
matplotlib.use("Agg")  # safe for CLI/WSL
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator


# -------- paths (edit if needed) --------
PATHS = {
    "iq_chain": "exp01_ibm_baseline+agent__iq_chain__mean_abs_err_MHz(1).csv",
    "chain": "exp01_ibm_baseline+agent__chain__mean_abs_err_MHz(1).csv",
    "realistic_chain": "exp01_ibm_baseline+agent__realistic_chain__mean_abs_err_MHz(1).csv",
}

OUT_PNG = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol.png"
OUT_PDF = "exp01_all_noise_agent1_vs_baselines_mae_vs_nl_ijcai_singlecol.pdf"


def load_table(csv_path: str):
    """first col is method name; remaining cols are noise levels (nl)."""
    df = pd.read_csv(csv_path)
    id_col = df.columns[0]
    x_cols = [c for c in df.columns if c != id_col]

    xs = np.array(sorted([float(c) for c in x_cols]))
    col_map = {float(c): c for c in x_cols}
    x_cols_ordered = [col_map[x] for x in xs]

    labels = df[id_col].astype(str).tolist()
    series = {
        lab: df.loc[i, x_cols_ordered].astype(float).to_numpy()
        for i, lab in enumerate(labels)
    }
    return xs, labels, series


def pick_agent_label(labels):
    for cand in labels:
        if cand.lower() in {"agent1", "a1", "onlyagent", "agent"}:
            return cand
    return "Agent1" if "Agent1" in labels else None


def main():
    # IJCAI-ish typography, but slightly larger for readability
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "dejavuserif",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.9,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,
    })

    tables = {k: load_table(v) for k, v in PATHS.items()}

    # Global limits
    global_ymax = 0.0
    x_min, x_max = None, None
    for _, (xs, _, series) in tables.items():
        global_ymax = max(global_ymax, max(float(np.nanmax(y)) for y in series.values()))
        x_min = float(np.min(xs)) if x_min is None else min(x_min, float(np.min(xs)))
        x_max = float(np.max(xs)) if x_max is None else max(x_max, float(np.max(xs)))

    # stable markers
    all_labels = []
    for _, (_, labels, _) in tables.items():
        for l in labels:
            if l not in all_labels:
                all_labels.append(l)
    marker_cycle = ["o", "s", "^", "v", "P", "X", "D", "<", ">"]
    mk = {lab: marker_cycle[i % len(marker_cycle)] for i, lab in enumerate(all_labels)}
    mk["Agent1"] = "D"

    # ---- single-column figure size ----
    # IJCAI single column is ~3.25 in wide; use ~3.35 in for a tiny margin.
    # Height: 3 stacked panels + legend space
    fig, axes = plt.subplots(
        3, 1, figsize=(3.35, 6.2), sharex=True, sharey=True
    )

    panel_order = ["iq_chain", "chain", "realistic_chain"]
    panel_titles = {"iq_chain": "iq_chain", "chain": "chain", "realistic_chain": "realistic_chain"}
    panel_letters = ["(a)", "(b)", "(c)"]

    handles_for_legend = {}

    # style knobs (bigger than before)
    baseline_lw = 1.2
    baseline_ms = 3.4
    baseline_alpha = 0.75

    agent_lw = 2.3
    agent_ms = 4.0

    label_fs = 9
    tick_fs = 8
    title_fs = 9
    legend_fs = 7.8

    for ax, key, letter in zip(axes, panel_order, panel_letters):
        xs, labels, series = tables[key]
        agent_label = pick_agent_label(labels)
        baseline_labels = [l for l in labels if l != agent_label]

        # baselines
        for lab in baseline_labels:
            line, = ax.plot(
                xs, series[lab],
                marker=mk.get(lab, "o"),
                linewidth=baseline_lw,
                markersize=baseline_ms,
                alpha=baseline_alpha,
                label=lab
            )
            if lab not in handles_for_legend:
                handles_for_legend[lab] = line

        # agent
        if agent_label is not None:
            line, = ax.plot(
                xs, series[agent_label],
                marker=mk.get("Agent1", "D"),
                linewidth=agent_lw,
                markersize=agent_ms,
                label="Agent1"
            )
            if "Agent1" not in handles_for_legend:
                handles_for_legend["Agent1"] = line

        # panel label + title (compact)
        ax.text(0.01, 0.98, f"{letter} {panel_titles[key]}",
                transform=ax.transAxes, ha="left", va="top", fontsize=title_fs)

        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
        ax.yaxis.set_minor_locator(AutoMinorLocator(2))
        ax.tick_params(axis="both", which="major", length=3.2, pad=1.2, labelsize=tick_fs)
        ax.tick_params(axis="both", which="minor", length=1.8)

        ax.grid(True, which="major", linestyle="--", linewidth=0.55, alpha=0.35)
        ax.grid(True, which="minor", linestyle=":", linewidth=0.45, alpha=0.25)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(0, global_ymax * 1.05)

    # common labels
    axes[1].set_ylabel("MAE", fontsize=label_fs, labelpad=3)
    axes[-1].set_xlabel("Noise level (nl)", fontsize=label_fs, labelpad=2)

    # ---- legend at bottom (bigger + multi-column) ----
    legend_labels = list(handles_for_legend.keys())
    if "Agent1" in legend_labels:
        legend_labels.remove("Agent1")
        legend_labels = ["Agent1"] + sorted(legend_labels)
    else:
        legend_labels = sorted(legend_labels)

    legend_handles = [handles_for_legend[l] for l in legend_labels]

    leg = fig.legend(
        legend_handles, legend_labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=2,                 # you can change to 3 if you want it shorter
        fontsize=legend_fs,
        frameon=True,
        fancybox=False,
        framealpha=0.95,
        borderpad=0.3,
        handlelength=1.8,
        handletextpad=0.5,
        columnspacing=0.9
    )
    leg.get_frame().set_linewidth(0.7)

    # leave space for legend
    fig.tight_layout(pad=0.4, rect=(0, 0.05, 1, 1))

    fig.savefig(OUT_PNG, dpi=600, bbox_inches="tight")
    fig.savefig(OUT_PDF, bbox_inches="tight")
    plt.close(fig)

    print("Saved:")
    print(" -", OUT_PNG)
    print(" -", OUT_PDF)


if __name__ == "__main__":
    main()
